In [1]:
import sys
import subprocess
import pkg_resources
import numpy as np

required = { 'scikit-image'}
installed = {pkg.key for pkg in pkg_resources.working_set}
missing = required - installed

if missing:
    python = sys.executable
    subprocess.check_call([python, '-m', 'pip', 'install', *missing], stdout=subprocess.DEVNULL)

C:\Users\s473593\AppData\Local\temp\ipykernel_8608\2649920712.py:3: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


###### 1. Opis danych dla pierwszych 6 zajęć laboratoryjnych

Dane są dostępne w postaci spakowanej w pliku <a href="http://fraktal.faculty.wmi.amu.edu.pl/symulowanie_wizualne/train_test_sw.zip">train_test_sw.zip</a>. Po rozpakowaniu otrzymujmemy katalog <i>train_test_sw</i> zawierający:

<ol>
        <li>katalog <i>train_sw</i> - katalog z listą podkatalogów przechowujących obrazy treningowe (format 4-bajtowy rgba) podzielone na kategorie - nazwa katalogu odpowiada nazwie kategorii
     <li>katalog <i>test_sw</i> - katalog z listą plików zawierających obrazy testowe   
     <li>plik <i>test_labels.json</i> - zawiera etykiety obrazów testowych  w formacie JSON - pojedyncza dana to słownik  
         <p>
             <code>
                 {
                    "filename": nazwa pliku,
                    "value": nazwa_kategorii
                 }
            </code>
 </ol>

#### 2. Funkcja <code>load_train_data(input_dir, newSize=(64,64))</code>

Funkcja ta wczytuje dane teningowe, przeskalowuje je do rozmiaru podanego w drugim parametrze (z interpolacją typu cv2.INTER_AREA), przeprowadza ich normalizację  i zwraca słownik w formacie:
<p>
<code>
{  "data" - lista (typ <code>list</code>) obrazów, gdzie pojedynczy obraz jest tablicą <code>numpy.array</code> zapisaną wierszowo blokami rgb
   "categories_name" - lista nazw kategorii klasyfikacyjnych, gdzie pojedyncza nazwa kategorii jest typu <code>str</code>
   "categories_count" - lista ilości przykładów z pozszczególnych kategorii w danych treningowych 
   "labels" - lista etykiet wszystkich danych treningowych
 }
 </code>
 
 Parametry:
    
 - <code>input_dir</code> przyjmuje wartość  ścieżki katalogu 'train_sw'
 - <code>newSize</code> przyjmuje wartość  krotki określającej rozmiar przeskalowanych obrazów   

In [2]:


def load_train_data(input_dir, newSize=(64,64)):
    '''
    '''

    
    import numpy as np
    import pandas as pd
    import os
    from skimage.io import imread
    import cv2 as cv
    from pathlib import Path
    import random
    from shutil import copyfile, rmtree
    import json

    import matplotlib
    
    image_dir = Path(input_dir)
    categories_name = []
    for file in os.listdir(image_dir):
        d = os.path.join(image_dir, file)
        if os.path.isdir(d):
            categories_name.append(file)

    folders = [directory for directory in image_dir.iterdir() if directory.is_dir()]

    train_img = []
    categories_count=[]
    labels=[]
    for i, direc in enumerate(folders):
        count = 0
        for obj in direc.iterdir():
            if os.path.isfile(obj) and os.path.basename(os.path.normpath(obj)) != 'desktop.ini':
                labels.append(os.path.basename(os.path.normpath(direc)))
                count += 1
                img = imread(obj)#zwraca ndarry postaci xSize x ySize x colorDepth
                img = cv.resize(img, newSize, interpolation=cv.INTER_AREA)# zwraca ndarray
                img = img / 255#normalizacja
                train_img.append(img)
        categories_count.append(count)
    X={}
    X["values"] = np.array(train_img)
    X["categories_name"] = categories_name
    X["categories_count"] = categories_count
    X["labels"]=labels
    return X
    


In [3]:
data = load_train_data('../train_test_sw/train_sw')
print(data['categories_name'])
print(data['categories_count'])
print([data["labels"][50],data["labels"][200],data["labels"][300]])
print(list(data["values"][100][1][10]))

['Beech', 'Gardenia', 'Lemon', 'Mean', 'Tomato']
[203, 206, 206, 206, 206]
['Beech', 'Beech', 'Gardenia']
[np.float64(0.3843137254901961), np.float64(0.08627450980392157), np.float64(0.11764705882352941), np.float64(1.0)]


#### 3. Funkcja <code>load_test_data(input_dir, newSize=(64,64))</code>

Funkcja ta  wczytuje dane testowe, przeskalowuje je do rozmiaru podanego w drugim parametrze (z 
interpolacją typu cv2.INTER_AREA), przeprowadza ich normalizację  i zwraca słownik w formacie:
<p>
    
<code>
{  "data" - lista (typ <code>list</code>) obrazów, gdzie pojedynczy obraz jest tablicą <code>numpy.array</code> zapisaną wierszowo blokami rgb
   "categories_name" - lista nazw kategorii klasyfikacyjnych, gdzie pojedyncza nazwa kategorii jest typu <code>str</code>
   "categories_count" - lista ilości przykładów z pozszczególnych kategorii w danych testowych 
   "labels" - lista etykiet wszystkich danych testowych
 }
 </code>
 
 Parametry:
    
 - <code>input_dir</code> przyjmuje wartość  ścieżki katalogu 'test_sw'
 - <code>newSize</code> przyjmuje wartość  krotki określającej rozmiar przeskalowanych obrazów   


In [4]:
def load_test_data(input_dir, newSize=(64,64)):
    '''
    '''

    import numpy as np
    import pandas as pd
    import os
    from skimage.io import imread
    import cv2 as cv
    from pathlib import Path
    import random
    from shutil import copyfile, rmtree
    import json



    image_path = Path(input_dir)

    labels_path = image_path.parents[0] / 'test_labels.json'

    #with labels_path.open("r", encoding ="utf-8") as f:
    jsonString = labels_path.read_text()
    objects = json.loads(jsonString)

    #print(objects)

    categories_name = []
    categories_count=[]
    count = 0
    c = objects[0]['value']
    for e in  objects:
        if e['value'] != c:
            #print(count)
            #print(c)
            categories_count.append(count)
            c = e['value']
            count = 1
        else:
            count += 1
        if not e['value'] in categories_name:
            categories_name.append(e['value'])

    categories_count.append(count)



    test_img = []

    labels=[]
    for e in objects:
        p = image_path / e['filename']
        img = imread(p)#zwraca ndarry postaci xSize x ySize x colorDepth
        img = cv.resize(img, newSize, interpolation=cv.INTER_AREA)# zwraca ndarray
        img = img / 255#normalizacja
        test_img.append(img)
        labels.append(e['value'])


    X={}
    X["values"] = np.array(test_img)
    X["categories_name"] = categories_name
    X["categories_count"] = categories_count
    X["labels"]=labels
    return X



In [5]:
data = load_test_data('../train_test_sw/test_sw')
print(data['categories_name'])
print(data['categories_count'])
print([data["labels"][10],data["labels"][20],data["labels"][30]])
print(list(data["values"][10][1][10]))

['Beech', 'Gardenia', 'Lemon', 'Mean', 'Tomato']
[51, 52, 52, 52, 52]
['Beech', 'Beech', 'Beech']
[np.float64(0.058823529411764705), np.float64(0.00784313725490196), np.float64(0.0), np.float64(1.0)]


#### 4.  Zadanie 1 (4pkt):

Napisz kod klasy <code>KNearestNeighbor</code> implementującej klasyfikator <i>knn</i>. Należy zimplementować następujące  metody:


 - <code>konstruktor</code> pobierający listę obrazów treningowych (zgodną zw składową 'values' wczytanego słownika) oraz listę ich etykiet
 - metoda <code>l_p_metric(image1, image2, p):</code> zwracająca wartość odległości pomiędzy dwoma obrazami, mierzoną normą typu <i>l_p</i> - parametr <code>p</code> określa 'potęgę' normy
 - metoda <code>predict(test_images, k,p):</code> zwracająca listę prognozowanych etykiet dla obrazów testowych (parametr <code>test_images</code>). Paramter drugi określa liczbę przeszukiwanych sąsiadów, natomiast paramter trzeci określa potęgę wybranej metryki.
 - metoda <code>accuracy(test_images, k,p)</code> zwracająca dokładność klasyfikatora na zbiorze testowym. Parametr drugi i trzeci są jak w metodzie <code>predict()</code>


In [6]:
class KNearestNeighbor(object):
    # BEGIN SOLUTION 
    def __init__(self, images, labels):
        self.train_images = np.array([img.flatten() for img in images])
        self.train_labels = np.array(labels)

    def predict(self, test_images: np.array, K: int, P: float):
        predictions = []
        for test_img in test_images:
            flat_test_img = test_img.flatten()
            distances = [self.l_p_metric(flat_test_img, train_img, P)
                         for train_img in self.train_images]

            neighbors_indices = np.argsort(distances)[:K]

            neighbors_labels = self.train_labels[neighbors_indices]

            unique_labels, counts = np.unique(neighbors_labels, return_counts=True)
            majority_label = unique_labels[np.argmax(counts)]

            predictions.append(majority_label)

        return predictions

    def l_p_metric(self, image1: np.array, image2: np.array, p: float):
        return np.sum(np.abs(image1 - image2) ** p) ** (1.0 / p)

    def accuracy(self, test_labels, predicted_labels ):
        correct = np.sum(np.array(predicted_labels) == np.array(test_labels))
        return correct / len(test_labels)
    
    # END SOLUTION


In [7]:
from sklearn.preprocessing import LabelEncoder

size = (32, 32)

data_train = load_train_data("../train_test_sw/train_sw",newSize=size)
X_train = data_train['values']
y_train = data_train['labels']


data_test = load_test_data("../train_test_sw/test_sw",newSize=size)
X_test = data_test['values']
y_test = data_test['labels']

class_le = LabelEncoder()
y_train_enc = class_le.fit_transform(y_train)

y_test_enc = class_le.fit_transform(y_test)

In [102]:
Knn  = KNearestNeighbor(X_train, y_train_enc)

In [103]:
pred = Knn.predict(X_test, K=1,P=1)
print(Knn.accuracy(y_test_enc,pred))

pred = Knn.predict(X_test, K=1,P=2)
print(Knn.accuracy(y_test_enc,pred))

pred = Knn.predict(X_test, K=5,P=1)
print(Knn.accuracy(y_test_enc,pred))

pred = Knn.predict(X_test, K=5,P=2)
print(Knn.accuracy(y_test_enc,pred))

pred = Knn.predict(X_test, K=10,P=1)
print(Knn.accuracy(y_test_enc,pred))

pred = Knn.predict(X_test, K=10,P=2)
print(Knn.accuracy(y_test_enc,pred))

0.640926640926641
0.5868725868725869
0.5752895752895753
0.5752895752895753
0.5444015444015444
0.5173745173745173


####  5. Zadanie 2 (2pkt): 

Napisz kod funkcji  <code>crossValidation(X,y, n = 10,  k=1,p=1  ):</code> obliczającą algorytm <code>knn </code> z n-krotną walidacją krzyżową.

In [104]:
from itertools import chain

def crossValidation(X: np.array, y: np.array, n = 10,  k=1,p=1  ):
        '''
        :param X: train data
        :param y: encoded labels of train data
        :param n: value for n-fold cross validation
        :param k: k value  for knn
        :param p: p value for l_p metric
        :return:
        '''

        # BEGIN SOLUTION 
        indices = np.arange(len(X))
        np.random.shuffle(indices)

        X = X[indices]
        y = y[indices]

        fold_size = len(X) // n
        accuracies = []

        for i in range(n):
            start = i * fold_size
            end = min((i + 1) * fold_size, len(X))

            X_test = X[start:end]
            y_test = y[start:end]

            X_train = np.concatenate((X[:start], X[end:]), axis=0)
            y_train = np.concatenate((y[:start], y[end:]), axis=0)

            classifier = KNearestNeighbor(X_train, y_train)

            y_pred = classifier.predict(X_test, K=k, P=p)

            fold_accuracy = classifier.accuracy(y_test, y_pred)
            accuracies.append(fold_accuracy)

        return np.mean(accuracies)

        #END SOLUTION 


In [105]:

res = crossValidation(X_train, y_train_enc,n=len(X_train), k=5,p=1)
print(res)


0.5929892891918208


#### 6.  Zadanie 3 (4pkt): 

Napisz kod klasy <code>LogisticRegression</code> implementującej klasyfikator <i>wieloklasowej regresji logistycznej</i> z funkcją <code>softmax()</code> (ze standardowymi nazwami dwóch kluczowych funkcji: <i>fit()</i>, <i>predict()</i>).  Zastosuj ten kod do pobranych danych (zbiór walidacyjny losujemy ze zbioru treningowego) - oblicz następujące charakterystyki modelu dla danych walidacyjnych oraz treningowych: dokładność (accuracy), precyzję (precision), czułość(recall) oraz F1 - dla poszczególnych klas oraz globalnie (zob. np. <a href="https://medium.com/synthesio-engineering/precision-accuracy-and-f1-score-for-multi-label-classification-34ac6bdfb404">tu</a>).


In [8]:
import pandas as pd
from typing import List, Tuple, Callable, Optional
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [9]:
X_train = np.array([img.flatten() for img in X_train])
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train_enc, test_size=0.2, random_state=42)

In [33]:
# BEGIN SOLUTION (zad. 3)
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


class LogisticRegression:
    def __init__(self, alpha=0.01, max_iter=1000, eps=1e-5):
        self.alpha = alpha
        self.max_iter = max_iter
        self.eps = eps
        
    def calc_loss(self, y, yhat):
        return  -np.sum(y * np.log(yhat + 1e-7)) / y.shape[0]
    
    def calc_dloss(self, X, y, Yhat):
        return np.dot(X.T, Yhat - y) / y.shape[0]

    def fit(self, X, y):
        X = np.array(np.hstack([np.ones((X.shape[0], 1)), X]))
        y_one_hot = pd.get_dummies(y).astype(float).values
        n_samples, n_features = X.shape
        n_classes = y_one_hot.shape[1]
        self.weights = np.random.randn(n_features, n_classes) * 0.01
        loss = 1000000 # just big initial loss
        for i in range(self.max_iter):
            Yhat = softmax(np.dot(X, self.weights))
            loss, prev_loss = self.calc_loss(y_one_hot, Yhat), loss
            self.weights -= self.alpha * self.calc_dloss(X, y_one_hot, Yhat)
            if np.abs(loss - prev_loss) < self.eps:
                break

    def predict(self, X):
        X = np.array(np.hstack([np.ones((X.shape[0], 1)), X]))
        z = np.dot(X, self.weights)
        probabilities = softmax(z)
        return np.argmax(probabilities, axis=1), probabilities

#END SOLUTION

lr_model = LogisticRegression(alpha=0.01, max_iter=50000, eps=5e-7)
lr_model.fit(X_train, y_train)

In [29]:
def eval(model, X, y):
    preds, _ = model.predict(X)
    precision, recall, f1, _ = precision_recall_fscore_support(y, preds, average=None)
    accuracy = accuracy_score(y, preds)

    mean_precision = np.mean(precision)
    mean_recall = np.mean(recall)
    mean_f1 = np.mean(f1)

    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        print(f"Class {i}: Precision: {p:.2f}, Recall: {r:.2f}, F1: {f:.2f}")

    print("--------------------")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Mean precision: {mean_precision:.2f}")
    print(f"Mean recall: {mean_recall:.2f}")
    print(f"Mean F1: {mean_f1:.2f}")

In [34]:
print("charakterystyki modelu dla danych treningowych:")
eval(lr_model, X_train, y_train)
print("charakterystyki modelu dla danych walidacyjnych:")
eval(lr_model, X_val, y_val)

charakterystyki modelu dla danych treningowych:
Class 0: Precision: 0.94, Recall: 0.88, F1: 0.91
Class 1: Precision: 0.94, Recall: 0.63, F1: 0.76
Class 2: Precision: 0.76, Recall: 0.82, F1: 0.79
Class 3: Precision: 0.75, Recall: 0.71, F1: 0.73
Class 4: Precision: 0.54, Recall: 0.72, F1: 0.62
--------------------
Accuracy: 0.75
Mean precision: 0.78
Mean recall: 0.75
Mean F1: 0.76
charakterystyki modelu dla danych walidacyjnych:
Class 0: Precision: 0.88, Recall: 0.90, F1: 0.89
Class 1: Precision: 0.69, Recall: 0.44, F1: 0.54
Class 2: Precision: 0.63, Recall: 0.77, F1: 0.69
Class 3: Precision: 0.58, Recall: 0.55, F1: 0.56
Class 4: Precision: 0.44, Recall: 0.53, F1: 0.48
--------------------
Accuracy: 0.64
Mean precision: 0.64
Mean recall: 0.64
Mean F1: 0.63


#### 7.  Zadanie 4 (1pkt): 

Oblicz ile danych z poszczególnych klas znajduje się po dodatniej/ujemnej stronie hiperpłaszczyzny klasyfikacyjnej dla danej klasy

In [44]:
def sigmoid(z):
    """Return the logistic function sigma(z) = 1/(1+exp(-z))."""
    return 1 / (1 + np.exp(-z))

def sigmoid_predict(X, w):
    X = np.array(np.hstack([np.ones((X.shape[0], 1)), X]))
    z = np.dot(X, w)
    return sigmoid(z)

In [68]:
for class_idx in range(5):
    print(f"-------------------------------------------------------")
    print(f"Dla hiperpłaszczyzny klasyfikacyjnej klasy {class_idx}:")
    # Here `lr_model.weights[:, class_idx]` represents theta vector for single-class binary classification
    preds = sigmoid_predict(X_train, lr_model.weights[:, class_idx])
    positive_negative_per_class = {
        c: (np.sum((y_train == c) & (preds > 0.5)), np.sum((y_train == c)))
        for c in np.unique(y_train)
    }
    for c, pos_neg in positive_negative_per_class.items():
        print(f"{c}: {pos_neg[0]} positive from {pos_neg[1]}")

-------------------------------------------------------
Dla hiperpłaszczyzny klasyfikacyjnej klasy 0:
0: 161 positive from 164
1: 44 positive from 161
2: 43 positive from 162
3: 10 positive from 166
4: 12 positive from 168
-------------------------------------------------------
Dla hiperpłaszczyzny klasyfikacyjnej klasy 1:
0: 87 positive from 164
1: 154 positive from 161
2: 96 positive from 162
3: 130 positive from 166
4: 136 positive from 168
-------------------------------------------------------
Dla hiperpłaszczyzny klasyfikacyjnej klasy 2:
0: 58 positive from 164
1: 65 positive from 161
2: 162 positive from 162
3: 20 positive from 166
4: 58 positive from 168
-------------------------------------------------------
Dla hiperpłaszczyzny klasyfikacyjnej klasy 3:
0: 48 positive from 164
1: 95 positive from 161
2: 35 positive from 162
3: 160 positive from 166
4: 138 positive from 168
-------------------------------------------------------
Dla hiperpłaszczyzny klasyfikacyjnej klasy 4:
0: 

#### 8. Dwa uzupełnienia

W pliku [lab1_add](lab1_add.ipynb) znajdują się  minimalne podstawy teoretyczne związane z konstrukcją funkcji kosztu oraz algorytmami optymalizacji.